In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

In [ ]:
import requests


def listar_todas_empresas(secret_key: str):
    url = "https://api.acessorias.com/companies/ListAll/"

    headers = {
        "Authorization": f"Bearer {secret_key}"
    }

    todas_empresas = []
    pagina = 1

    while True:
        params = {
            "obligations": "",
            "departments": "",
            "stateRegistrations": "",
            "registrationData": "",
            "contacts": "",
            "Pagina": pagina
        }

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=60
        )

        print(f"Buscando página {pagina} - Status: {response.status_code}")

        response.raise_for_status()

        dados = response.json()

        # Caso a API retorne lista diretamente
        if isinstance(dados, list):
            empresas = dados

        # Caso a API retorne dentro de alguma chave
        elif isinstance(dados, dict):
            empresas = (
                dados.get("data")
                or dados.get("empresas")
                or dados.get("companies")
                or dados.get("result")
                or []
            )

        else:
            empresas = []

        if not empresas:
            print("Nenhum registro encontrado. Fim da paginação.")
            break

        todas_empresas.extend(empresas)

        pagina += 1

    return todas_empresas


secret_key = "bd88bd028e835296f77887fd6fdd9c19"

empresas = listar_todas_empresas(secret_key)

print(f"Total de empresas encontradas: {len(empresas)}")

for empresa in empresas:
    display(empresa)

In [ ]:
from docx import Document
import pandas as pd

arquivo = r"C:\Users\manja\Downloads\EXTRATO DE CONTAS ABRIL 2026.docx"

doc = Document(arquivo)

tabelas_extraidas = []

for tabela in doc.tables:
    dados = []

    for linha in tabela.rows:
        valores = [celula.text.strip() for celula in linha.cells]
        dados.append(valores)

    df = pd.DataFrame(dados)
    tabelas_extraidas.append(df)

# Exemplo: mostrar a primeira tabela
display(tabelas_extraidas[0])

In [1]:
import os
import json
import base64
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.utils.helpers import read_pdfs

load_dotenv()

controle_pdf = Path("pdfs_baixados.json")
downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)

BASE_URL = os.getenv("EVOLUTION_BASE_URL", "https://api-whatsapp-xoor.onrender.com/").rstrip("/")
API_KEY = os.getenv("EVOLUTION_API_KEY")
INSTANCE_NAME = os.getenv("EVOLUTION_INSTANCE_NAME", "ExtractPDFs")


In [2]:
class WhatsAppChat():
    def __init__(self, base_url, api_key, instance_name):
        self.base_url = base_url
        self.api_key = api_key
        self.instance_name = instance_name
        self.headers = {"apikey": self.api_key, "Content-Type": "application/json"}
        self.downloads_dir = Path("downloads")
        self.downloads_dir.mkdir(parents=True, exist_ok=True)
        self.controle_pdf = Path("controle_pdf.json")

    def read_pdfs(self):
        if not self.controle_pdf.exists():
            return {}

        with open(self.controle_pdf, "r", encoding="utf-8") as file:
            conteudo = file.read().strip()

            if not conteudo:
                return {}

            try:
                return json.loads(conteudo)
            except json.JSONDecodeError:
                return {}
            
    def save_pdfs_control(self, data):
        self.controle_pdf.parent.mkdir(parents=True, exist_ok=True)

        with open(self.controle_pdf, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2, ensure_ascii=False)

    def connect(self):
        response = requests.post(
            f"{self.base_url}/chat/findChats/{self.instance_name}",
            headers=self.headers,
            json={},
            timeout=60
        )

        response.raise_for_status()
        return response.json()
    
    def groups(self):
        grupos = []
        chats = self.connect()

        for chat in chats:
            remote_jid = chat.get("remoteJid")

            if remote_jid and remote_jid.endswith("@g.us"):
                grupos.append(chat)

        return grupos
    
    def findGroupName(self, grupos, group_name):
        for grupo in grupos:
            if grupo.get("pushName") == group_name or grupo.get("name") == group_name:
                return grupo.get("remoteJid")

        return None
    
    def mensagens(self, remote_jid):
        url_messages = f"{self.base_url}/chat/findMessages/{self.instance_name}"

        payload = {
            "where": {
                "key": {
                    "remoteJid": remote_jid
                }
            },
            "limit": 50
        }

        response = requests.post(
            url_messages,
            headers=self.headers,
            json=payload,
            timeout=60
        )

        response.raise_for_status()
        return response.json()
    
    def download_pdf(self, messages_response):
        pdfs_baixados = self.read_pdfs()

        self.downloads_dir.mkdir(parents=True, exist_ok=True)

        url_media = f"{self.base_url}/chat/getBase64FromMediaMessage/{self.instance_name}"

        records = (
            messages_response
            .get("messages", {})
            .get("records", [])
        )

        total_baixados = 0
        total_ignorados = 0
        total_erros = 0
        total_indisponiveis = 0

        for mensagem in records:
            key = mensagem.get("key", {})
            message_id = key.get("id")
            message_type = mensagem.get("messageType")

            if not message_id:
                total_ignorados += 1
                continue

            if message_type != "documentMessage":
                total_ignorados += 1
                continue

            document = (
                mensagem
                .get("message", {})
                .get("documentMessage", {})
            )

            mimetype = document.get("mimetype")
            nome_file = document.get("fileName") or f"{message_id}.pdf"

            if mimetype != "application/pdf":
                total_ignorados += 1
                continue

            if message_id in pdfs_baixados:
                print(f"PDF {nome_file} já processado, pulando...")
                total_ignorados += 1
                continue

            print("-" * 80)
            print("Message ID:", message_id)
            print("Arquivo:", nome_file)
            print("Grupo/Chat:", key.get("remoteJid"))

            payload_media = {
                "message": mensagem
            }

            try:
                response_media = requests.post(
                    url_media,
                    headers=self.headers,
                    json=payload_media,
                    timeout=120
                )

                print("Status download:", response_media.status_code)

                # Status 400 na Evolution geralmente indica que a mídia
                # não está mais disponível para download, expirou, foi removida
                # ou a stream não pôde ser buscada.
                # Nesse caso, registra no controle e segue para o próximo PDF.
                if response_media.status_code == 400:
                    print(f"PDF indisponível para download, pulando: {nome_file}")
                    print("Resposta da API:")
                    print(response_media.text)

                    pdfs_baixados[message_id] = {
                        "file_name": nome_file,
                        "mimetype": mimetype,
                        "remote_jid": key.get("remoteJid"),
                        "participant": key.get("participant"),
                        "from_me": key.get("fromMe"),
                        "push_name": mensagem.get("pushName"),
                        "message_timestamp": mensagem.get("messageTimestamp"),
                        "local_path": None,
                        "status": "indisponivel_400",
                        "error_response": response_media.text
                    }

                    self.save_pdfs_control(pdfs_baixados)

                    total_indisponiveis += 1
                    continue

                if response_media.status_code >= 400:
                    print("Erro ao baixar mídia:")
                    print(response_media.text)
                    total_erros += 1
                    continue

                media_response = response_media.json()

                media_base64 = (
                    media_response.get("base64")
                    or media_response.get("data", {}).get("base64")
                )

                if not media_base64:
                    print(f"Base64 não encontrado para o arquivo: {nome_file}")
                    print(media_response)
                    total_erros += 1
                    continue

                if media_base64.startswith("data:"):
                    media_base64 = media_base64.split(",", 1)[1]

                file_bytes = base64.b64decode(media_base64)

                file_path = self.downloads_dir / nome_file
                file_path.write_bytes(file_bytes)

                pdfs_baixados[message_id] = {
                    "file_name": nome_file,
                    "mimetype": mimetype,
                    "remote_jid": key.get("remoteJid"),
                    "participant": key.get("participant"),
                    "from_me": key.get("fromMe"),
                    "push_name": mensagem.get("pushName"),
                    "message_timestamp": mensagem.get("messageTimestamp"),
                    "local_path": str(file_path.resolve()),
                    "status": "baixado"
                }

                self.save_pdfs_control(pdfs_baixados)

                total_baixados += 1

                print(f"PDF baixado: {nome_file}")
                print(f"Caminho: {file_path.resolve()}")

            except Exception as error:
                print(f"Erro inesperado ao baixar {nome_file}: {error}")
                total_erros += 1
                continue

        print("Download de PDFs concluído.")

        return {
            "status": "ok",
            "total_baixados": total_baixados,
            "total_ignorados": total_ignorados,
            "total_indisponiveis": total_indisponiveis,
            "total_erros": total_erros
        }
    
    
whats = WhatsAppChat(BASE_URL, API_KEY, INSTANCE_NAME)
grupos = whats.groups()
extrato = whats.findGroupName(grupos, "ROBO EXTRATO")
mensagens = whats.mensagens(extrato)
records = whats.download_pdf(mensagens)


PDF 0126_EXTBAN CORA_WFORTE.pdf já processado, pulando...
PDF 0126_EXTBAN CORA_WFORTE.pdf já processado, pulando...
PDF 0126_EXTBAN SICOOB 18.723-2_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN SICOOB 18.723-2_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTAPL RENT SICREDI_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTAPL RENT SICREDI_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTAPL SICREDI_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTAPL SICREDI_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN SICREDI 42480-4_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN SICREDI 42480-4_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN UNICRED 10.378-0_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN UNICRED 10.378-0_CEMAF OPE.pdf já processado, pulando...
PDF 0126_EXTBAN SICOOB 18713-5_ACR.pdf já processado, pulando...
PDF 0126_EXTBAN SICOOB 18713-5_ACR.pdf já processado, pulando...
PDF 0126_EXTCAP SICOOB_ACR.pdf já processado, pulando.